In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("GoldLayerAnalysis") \
    .getOrCreate()

In [8]:
BASE_PATH = "/home/jovyan/work/output"

fact_order_items = spark.read.csv(
    f"{BASE_PATH}/gold/fact_order_items",
    header=True,
    inferSchema=True
)

dim_products = spark.read.csv(
    f"{BASE_PATH}/gold/dim_products",
    header=True,
    inferSchema=True
)

In [9]:
fact_order_items.createOrReplaceTempView("fact_order_items")
dim_products.createOrReplaceTempView("dim_products")

In [10]:
query = """
WITH monthly_revenue AS (
    SELECT
        dp.product_category_name,
        YEAR(f.order_date) AS year,
        MONTH(f.order_date) AS month,
        COUNT(*) AS txn_count,
        SUM(f.price + f.freight_value) AS monthly_revenue
    FROM fact_order_items f
    JOIN dim_products dp
        ON f.product_key = dp.product_key
    GROUP BY dp.product_category_name, YEAR(f.order_date), MONTH(f.order_date)
),

filtered AS (
    SELECT *
    FROM monthly_revenue
    WHERE txn_count >= 10
),

ranked AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY year, month
            ORDER BY monthly_revenue DESC
        ) AS monthly_rank
    FROM filtered
),

top_categories AS (
    SELECT product_category_name
    FROM (
        SELECT
            product_category_name,
            SUM(monthly_revenue) AS total_revenue,
            RANK() OVER (ORDER BY SUM(monthly_revenue) DESC) AS rnk
        FROM filtered
        GROUP BY product_category_name
    ) t
    WHERE rnk <= 5
),

final AS (
    SELECT
        r.product_category_name,
        r.year,
        r.month,
        r.monthly_revenue,
        r.monthly_rank,

        (r.monthly_revenue -
            LAG(r.monthly_revenue) OVER (
                PARTITION BY r.product_category_name
                ORDER BY r.year, r.month
            )
        ) / LAG(r.monthly_revenue) OVER (
            PARTITION BY r.product_category_name
            ORDER BY r.year, r.month
        ) AS mom_growth_pct,

        AVG(r.monthly_revenue) OVER (
            PARTITION BY r.product_category_name
            ORDER BY r.year, r.month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rolling_3m_avg_revenue

    FROM ranked r
    JOIN top_categories t
        ON r.product_category_name = t.product_category_name
)

SELECT *
FROM final
ORDER BY product_category_name, year, month
"""

spark.sql(query).show(50, False)

+---------------------+----+-----+------------------+------------+---------------------+----------------------+
|product_category_name|year|month|monthly_revenue   |monthly_rank|mom_growth_pct       |rolling_3m_avg_revenue|
+---------------------+----+-----+------------------+------------+---------------------+----------------------+
|beleza_saude         |2016|10   |5493.379999999999 |3           |NULL                 |5493.379999999999     |
|beleza_saude         |2017|1    |14039.74          |2           |1.5557562011002337   |9766.56               |
|beleza_saude         |2017|2    |26130.08          |2           |0.861151274881159    |15221.066666666666    |
|beleza_saude         |2017|3    |29852.870000000003|4           |0.14247143521948652  |23340.896666666667    |
|beleza_saude         |2017|4    |26397.99          |3           |-0.11573024637162191 |27460.313333333335    |
|beleza_saude         |2017|5    |52801.020000000004|1           |1.0001909236271398   |36350.6266666666